# Sum-of-Squares Spectrum Amplification for Chemistry Problems

This notebook demonstrates how to compute ground-state energies for chemistry systems with quantum phase estimation (QPE) and sum-of-squares spectrum amplification (SOSSA). 

Quantum phase estimation (QPE) is a standard method for estimating the eigenvalues (energies) of a Hamiltonian encoded in a unitary operator. See the [stretched N<sub>2</sub> notebook](qpe_stretched_n2.ipynb) for an introduction on QPE with trotterization or linear combinations of unitaries (LCU).


In this notebook, you will:
1. factorize a standard chemistry Hamiltonian in different ways,
2. build SOSSA phase estimation circuits, 
3. validate the circuits on H$_2$, 
4. and estimate fault-tolerant resources.

Install the dependencies with:

```bash
pip install 'qdk-chemistry[jupyter,plugins,qre]'
```

In [ ]:
from pathlib import Path

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import AlgorithmRef, Hamiltonian, MajoranaMapping
from qdk_chemistry.utils import Logger
from utils.sossa_utils import (
    SOSSA_QPE_BUILDER,
    build_sossa_qpe_circuit,
    compiled_circuit_mapper,
    direct_circuit_mapper,
    effective_normalization,
    hartree_fock_state_preparation,
    heisenberg_queries,
    make_fake_hamiltonian,
    require_sossa_walk,
    simulation_qubit_estimate,
)

Logger.set_global_level(Logger.LogLevel.off)

## Why SOSSA is useful

Qubitization embeds the scaled Hamiltonian $H/\lambda$ in a unitary walk operator. For target precision $\sigma_E$, the required number of walk queries in QPE scales as

$$
p=\left\lceil\frac{\pi\lambda}{2\sigma_E}\right\rceil .
$$

The normalization $\lambda$ therefore has a direct effect on circuit depth: a larger normalization requires more queries.

SOSSA applies when the Hamiltonian can be shifted into a positive sum-of-squares form,

$$
H-E_{\mathrm{SOS}}=H_{\mathrm{sqrt}}^{\dagger}H_{\mathrm{sqrt}}.
$$

Instead of block-encoding $H$ directly, SOSSA block-encodes $H_{\mathrm{sqrt}}$. The square-root transformation spreads nearby energies farther apart near the bottom of the spectrum, reducing the queries needed for ground-state estimation.

## How to get the sum-of-squares form for chemistry hamiltonians

### DFTHC

The electronic-structure Hamiltonian in second quantization is

$$
H=\sum_{pq}h_{pq}E_{pq}
+\frac{1}{2}\sum_{pqrs}g_{pqrs}E_{pq}E_{rs}
+E_{\mathrm{nuc}},
\qquad
E_{pq}=\sum_{\sigma}a_{p\sigma}^{\dagger}a_{q\sigma}.
$$

The $O(N^4)$ tensor $g$ is expensive to use directly. Double-factorized tensor hypercontraction (DFTHC) replaces it with compact tensors of shape $(N,R,B,C)$ and rewrites the interaction as a sum of squared one-body operators,

$$
H_{\mathrm{DFTHC}}=\sum_{pq}h_{pq}^{(1)\prime}E_{pq}
+\frac{1}{2}\sum_{r,c}\left(W^{(rc)}\mathbb{I}
+\sum_b w_b^{(rc)}\sum_{pq}u_{b,p}^{(r)}u_{b,q}^{(r)}E_{pq}\right)^2+\text{constant}.
$$

Finding a compact, accurate DFTHC fit is a hard non-convex optimization. For validation, we load a converged H$_2$ factorization with $N=2$, $R=2$, $B=2$, and $C=1$.

In addition to DFTHC, this notebook will demonstrate how the same workflow for circuit generation, simulation and resource estimation works with the our open-source double factorization algorithm and synthetic data with a known number of orbitals and factorized parameters.

In [ ]:
json_path = Path("data") / "h2_dfthc_r2_b2_c1.hamiltonian.json"
hamiltonian = Hamiltonian.from_json(json_path.read_text())
num_alpha, num_beta = 1, 1
system_label = "H2 (stored DFTHC, N=2 R=2 B=2 C=1)"

### Double factorization as a special case for DFTHC

Conventional double factorization is the easy special case of DFTHC: $R=R_{\mathrm{DF}}$, $B=N$, and $C=1$. The following cell runs the classical workflow, computes a CASCI reference energy, and stores the factorized Hamiltonian under `n2_hamiltonian` for later resource estimation.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.data.symmetry import SymmetryLabel, axes
from qdk_chemistry.utils import compute_valence_space_parameters

structure = Structure.from_xyz_file(Path("data/stretched_n2.structure.xyz"))
e_hf, wfn_hf = create("scf_solver").run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz",
)

num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
valence_wfn = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o,
).run(wfn_hf)

valence_indices = valence_wfn.get_orbitals().active_indices()
localized_wfn = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(
    valence_wfn,
    list(valence_indices.indices(SymmetryLabel([axes.alpha()]))),
    list(valence_indices.indices(SymmetryLabel([axes.beta()]))),
)

hamiltonian_constructor = create("hamiltonian_constructor")
localized_hamiltonian = hamiltonian_constructor.run(localized_wfn.get_orbitals())
num_alpha_electrons, num_beta_electrons = localized_wfn.get_active_num_electrons()
_, selected_ci_wfn = create(
    "multi_configuration_calculator",
    "macis_asci",
    calculate_one_rdm=True,
    calculate_two_rdm=True,
).run(localized_hamiltonian, num_alpha_electrons, num_beta_electrons)

autocas_wfn = create("active_space_selector", "qdk_autocas_eos").run(
    selected_ci_wfn
)
active_hamiltonian = hamiltonian_constructor.run(autocas_wfn.get_orbitals())
n2_num_alpha, n2_num_beta = autocas_wfn.get_active_num_electrons()

n2_reference_energy, _ = create("multi_configuration_calculator", "macis_cas").run(
    active_hamiltonian, n2_num_alpha, n2_num_beta
)
print(f"Hartree-Fock energy: {e_hf:.6f} Hartree")
print(f"Active-space CASCI energy: {n2_reference_energy:.6f} Hartree")
print(f"Active space: {n2_num_alpha} alpha + {n2_num_beta} beta electrons")

factorizer = create("hamiltonian_factorization", "double_factorization")
factorizer.settings().set("truncation_threshold", 1e-8)
n2_hamiltonian = factorizer.run(active_hamiltonian)
n2_system_label = "stretched N2 (double factorization)"

### Synthesize factorized Hamiltonians from tensor shapes

When only the factorization dimensions of a chemical system are available, synthetic tensors can provide a rough circuit-resource estimate. The `make_fake_hamiltonian` helper samples a symmetric one-body matrix, normalized basis vectors $U[R,B,N]$, two-body weights $W[R,B,C]$, and identity weights $\bar{W}[R,C]$ using a fixed random seed. The resulting Hamiltonians reproduce the specified tensor dimensions, but not the physical spectra of Fe$_2$S$_2$ or FeMoCo.

In [ ]:
SYNTHETIC_SYSTEMS = {
    "Fe2S2 (30e, 20o)": {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    },
    "FeMoCo (54e, 54o)": {
        "electrons": 54,
        "N": 54,
        "R": 10,
        "B": 27,
        "C": 27,
        "lambda_eff": 21.3674,
        "b_coeff": 9,
        "b_rot": 16,
    },
}

SYNTHETIC_SEED = 42
synthetic_hamiltonians = {}
for name, parameters in SYNTHETIC_SYSTEMS.items():
    synthetic_hamiltonians[name] = make_fake_hamiltonian(
        n=parameters["N"],
        r=parameters["R"],
        b=parameters["B"],
        c=parameters["C"],
        seed=SYNTHETIC_SEED,
    )
    synthetic_container = synthetic_hamiltonians[name].get_container()
    print(
        f"{name}: (N, R, B, C) = "
        f"({synthetic_container.get_num_orbitals()}, "
        f"{synthetic_container.get_num_ranks()}, "
        f"{synthetic_container.get_num_bases()}, "
        f"{synthetic_container.get_num_copies()})"
    )

## SOSSA circuit building

Now we will show how the factorized hamiltonian from different sources can all go through the same workflow to get a sossa circuit. Each step along the way takes the classical hamiltonian into tangible quantum operations.

### Map the factorization to a sum-of-squares operator

The `sum_of_squares` mapper applies the Jordan-Wigner transformation and exposes the factorization as

$$
H=\sum_{\alpha}O_{\alpha}^{\dagger}O_{\alpha}+E_{\mathrm{SOS}}.
$$

The one-body generators come from diagonalizing the corrected matrix $h^{(1)\prime}$; the remaining generators are the squared spin-free factors, one for each $(r,c)$ pair. The constant $E_{\mathrm{SOS}}$ collects the terms that must be added back when a measured phase is converted to an energy.

In [ ]:
container = hamiltonian.get_container()
num_orbitals = container.get_num_orbitals()

operator = create("qubit_mapper", "sum_of_squares").run(
    hamiltonian,
    MajoranaMapping.jordan_wigner(2 * num_orbitals),
)
print(operator.get_container().get_summary())

### Build the $H_2$ SOSSA walk

The `sossa` builder block-encodes the square root of $H-E_{\mathrm{SOS}}$ and turns it into a walk whose phase can be measured by QPE. A classical reference energy is needed to derive the amplified normalization $\lambda_{\mathrm{eff}}$. The stored H$_2$ data does not include one, so this validation path reports the conservative raw normalization $\Lambda$ instead.

In [ ]:
sossa_unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "sossa")
walk = create("hamiltonian_unitary_builder", "sossa").run(operator)
walk_container = require_sossa_walk(walk)

lambda_effective, lambda_source = effective_normalization(walk_container)

print(walk_container.get_summary())
print(f"Lambda     = {walk_container.normalization:.6f} Hartree")
print(f"lambda_eff = {lambda_effective:.6f} Hartree ({lambda_source})")

## Compile the walk into gates

Unlike a flat LCU, DFTHC has a two-level index, so SOSSA uses two preparation stages:

- **Outer PREPARE** selects a generator $O_\alpha$.
- **Inner PREPARE** loads the signed, square-root-weighted terms inside that generator.
- **SELECT** rotates into the chosen orbital basis, applies the operator, and rotates back.

Reflections around these operations form the walk used by QPE.

### Two circuit mappings, one walk

| | `qre_mapper` | `validation_mapper` |
|---|---|---|
| PREPARE | Alias sampling | Dense/direct preparation |
| SELECT | QROM and phase gradient | Direct lookup |
| Use | Fault-tolerant resource estimate | State-vector validation |

The compiled mapper includes realistic arithmetic and ancillas but is too wide to simulate. The direct mapper keeps H$_2$ simulatable but is not a valid cost model. Both implement the same walk at the same rotation precision.

In [ ]:
VALIDATION_ROTATION_BITS = 15
validation_mapper = direct_circuit_mapper(
    rotation_bit_precision=VALIDATION_ROTATION_BITS
)

print("Validation mapper: dense state preparation + direct oracles")
print(f"  Givens angle precision: {VALIDATION_ROTATION_BITS} bits")
print("  No alias scratch or phase-gradient register")

## Prepare a trial state

QPE returns the target energy only when the input state overlaps the corresponding eigenstate. Here we use the canonical Hartree-Fock determinant, prepared with sparse isometry. Strongly correlated systems may need a multi-configuration state; the [stretched N<sub>2</sub> notebook](qpe_stretched_n2.ipynb) demonstrates that workflow.

In [ ]:
state_preparation = hartree_fock_state_preparation(hamiltonian, num_alpha, num_beta)

## Simulation and resource estimation

### Assemble the $H_2$ validation circuit

Unary-iteration QPE prepares a cosine-windowed phase register over $p+1$ slots and applies exactly $p$ walk queries. It is currently the only QPE builder that accepts a SOSSA walk. The qubit-count guard is evaluated before circuit construction so an unexpectedly large problem is never sent to the simulator.

In [ ]:
SIMULATION_QUERIES = 7
MAX_SIMULATION_QUBITS = 26

estimated_qubits = simulation_qubit_estimate(walk_container, SIMULATION_QUERIES)
validation_circuit = None
validation_builder = None

if estimated_qubits > MAX_SIMULATION_QUBITS:
    print(
        f"Skipping simulation: the walk needs about {estimated_qubits} qubits, over the "
        f"MAX_SIMULATION_QUBITS = {MAX_SIMULATION_QUBITS} cutoff."
    )
else:
    validation_settings = {
        "num_queries": SIMULATION_QUERIES,
        "circuit_mapper": validation_mapper,
        "unitary_builder": sossa_unitary_builder,
    }
    validation_builder = AlgorithmRef(
        "qpe_circuit_builder", SOSSA_QPE_BUILDER, **validation_settings
    )
    validation_circuit = create(
        "qpe_circuit_builder", SOSSA_QPE_BUILDER, **validation_settings
    ).run(
        state_preparation=state_preparation,
        qubit_hamiltonian=operator,
    )[0]
    print(
        f"Validation circuit: {validation_circuit.num_qubits} qubits, "
        f"{SIMULATION_QUERIES} queries"
    )

### Validate the $H_2$ circuit

The seven-query circuit is intentionally much shorter than a precision-scale calculation. It checks that the measured phase decodes to the expected energy through

$$
E=2\Lambda\cos^2(\pi\varphi)+E_{\mathrm{SOS}}.
$$

In [ ]:
SIMULATION_SHOTS = 50
SIMULATION_SEED = 42

if validation_circuit is None or validation_builder is None:
    print("Simulation was skipped by the qubit-count guard.")
else:
    from qdk.widgets import Histogram

    execution = create(
        "circuit_executor", "qdk_sparse_state_simulator", seed=SIMULATION_SEED
    ).run(validation_circuit, shots=SIMULATION_SHOTS)
    display(
        Histogram(
            bar_values={
                bitstring: count / execution.total_shots
                for bitstring, count in execution.bitstring_counts.items()
            }
        )
    )

    qpe = create("phase_estimation", SOSSA_QPE_BUILDER, shots=SIMULATION_SHOTS)
    qpe.settings().set("qpe_circuit_builder", validation_builder)
    qpe.settings().set(
        "circuit_executor",
        AlgorithmRef(
            "circuit_executor", "qdk_sparse_state_simulator", seed=SIMULATION_SEED
        ),
    )
    qpe_result = qpe.run(
        qubit_hamiltonian=operator, state_preparation=state_preparation
    )

    print(f"Measured phase fraction: {qpe_result.phase_fraction:.6f}")
    print(f"Decoded ground-state energy: {qpe_result.raw_energy:.6f} Hartree")
    print(f"Sign-branch candidates: {tuple(round(e, 6) for e in qpe_result.branching)}")

### Estimate resources for stretched $N_2$

The CASCI energy computed above supplies the low-energy promise for $\lambda_{\mathrm{eff}}$. We use it to size a unary-QPE circuit at millihartree precision, then map that logical circuit to a fault-tolerant Majorana architecture.

In [ ]:
from qdk.qre import estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

TARGET_PRECISION = 1e-3
MAX_ERROR = 0.01
N2_ROTATION_BITS = 15
N2_COEFFICIENT_BITS = 11

n2_container = n2_hamiltonian.get_container()
n2_num_orbitals = n2_container.get_num_orbitals()
n2_operator = create("qubit_mapper", "sum_of_squares").run(
    n2_hamiltonian,
    MajoranaMapping.jordan_wigner(2 * n2_num_orbitals),
)
n2_unitary_builder = AlgorithmRef(
    "hamiltonian_unitary_builder",
    "sossa",
    reference_ground_state_energy=n2_reference_energy,
)
n2_walk = create(
    "hamiltonian_unitary_builder",
    "sossa",
    reference_ground_state_energy=n2_reference_energy,
).run(n2_operator)
n2_walk_container = require_sossa_walk(n2_walk)
n2_lambda_effective, _ = effective_normalization(n2_walk_container)
n2_num_queries = heisenberg_queries(n2_lambda_effective, TARGET_PRECISION)

n2_qre_mapper = compiled_circuit_mapper(
    rotation_bit_precision=N2_ROTATION_BITS,
    coefficient_bit_precision=N2_COEFFICIENT_BITS,
)
n2_state_preparation = hartree_fock_state_preparation(
    n2_hamiltonian, n2_num_alpha, n2_num_beta
)
n2_qpe_builder = create(
    "qpe_circuit_builder",
    SOSSA_QPE_BUILDER,
    num_queries=n2_num_queries,
    circuit_mapper=n2_qre_mapper,
    unitary_builder=n2_unitary_builder,
)
n2_circuit = n2_qpe_builder.run(
    state_preparation=n2_state_preparation,
    qubit_hamiltonian=n2_operator,
)[0]

print(f"N2 lambda_eff: {n2_lambda_effective:.6f} Hartree")
print(f"N2 walk queries: {n2_num_queries}")
print(f"N2 logical qubits: {n2_circuit.num_qubits}")

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

n2_results = estimate(
    n2_circuit.get_qre_application(),
    architecture,
    isa_query,
    max_error=MAX_ERROR,
    name=f"SOSSA {n2_system_label}",
)
n2_results.add_factory_summary_column()
display(n2_results.as_frame())
plot_estimates(n2_results, figsize=(6, 4))

## Scope and limitations

- Producing an optimized DFTHC fit is out of scope; H$_2$ uses a stored fit and N$_2$ uses double factorization.
- The short H$_2$ simulation validates phase-to-energy decoding, not millihartree precision.
- Synthetic tensors provide realistic circuit dimensions but no physical spectrum, so they are resource-estimated and never simulated.

## Estimate resources for synthetic systems

Finally, build circuits for the published Fe$_2$S$_2$ and FeMoCo tensor shapes. The sweep uses an automatic memory/compute architecture in which 20% of the physical qubits support gates and the remainder use denser memory-oriented error correction.

In [ ]:
from qdk.qre import PSSPC, DynamicMemoryCompute, LatticeSurgery

MEMORY_COMPUTE_PERCENTAGE = 0.2
trace_query = (
    DynamicMemoryCompute.q(compute_capacity_percentage=MEMORY_COMPUTE_PERCENTAGE)
    * PSSPC.q()
    * LatticeSurgery.q()
)

sweep_results = []
for name, sweep_params in SYNTHETIC_SYSTEMS.items():
    sweep_circuit, _ = build_sossa_qpe_circuit(
        synthetic_hamiltonians[name],
        n_alpha=(sweep_params["electrons"] + 1) // 2,
        n_beta=sweep_params["electrons"] // 2,
        num_queries=heisenberg_queries(
            sweep_params["lambda_eff"], TARGET_PRECISION
        ),
        circuit_mapper=compiled_circuit_mapper(
            rotation_bit_precision=sweep_params["b_rot"],
            coefficient_bit_precision=sweep_params["b_coeff"],
        ),
    )
    sweep_result = estimate(
        sweep_circuit.get_qre_application(),
        architecture,
        isa_query,
        trace_query,
        max_error=MAX_ERROR,
        name=f"SOSSA {name} ({int(MEMORY_COMPUTE_PERCENTAGE * 100)}% compute)",
    )
    sweep_results.append(sweep_result)
    display(sweep_result.as_frame())

plot_estimates(sweep_results, figsize=(8, 5))